# 1. CLIP Linear Projection

Sử dụng một tầng `nn.Linear(clip_dim, clip_length * embedding_dim)` để chiếu không gian từ $512$ sang $10 \times 768 = 7680$.
Biến đổi hình dạng (`reshape`) vector phẳng thành chuỗi **Image Tokens** $[B, 10, 768]$ mà không dùng vòng lặp, tạo ra $10$ góc nhìn biểu diễn (learned subspace views) từ cùng một global feature của ảnh.

In [1]:
from __future__ import annotations

import torch
from torch import Tensor, nn
from src.config.clipcap_config import (
    CLIPCAP_MAPPER_CLIP_LENGTH,
    CLIPCAP_MAPPER_DROPOUT,
    CLIPCAP_MAPPER_FEEDFORWARD_DIM,
    CLIPCAP_MAPPER_FEEDFORWARD_MULTIPLIER,
    CLIPCAP_MAPPER_NUM_HEADS,
    CLIPCAP_MAPPER_NUM_LAYERS,
    CLIPCAP_MAPPER_PREFIX_LENGTH,
    CLIPCAP_PREFIX_INIT_MEAN,
    CLIPCAP_PREFIX_INIT_STD,
)


class ClipProjection(nn.Module):
    """Chiếu global CLIP feature thành chuỗi learned image tokens."""

    def __init__(
        self,
        clip_dim: int,
        embedding_dim: int,
        clip_length: int,
    ) -> None:
        super().__init__()
        for name, val in {
            "clip_dim": clip_dim,
            "embedding_dim": embedding_dim,
            "clip_length": clip_length,
        }.items():
            if isinstance(val, bool) or not isinstance(val, int) or val <= 0:
                raise ValueError(f"'{name}' phải là số nguyên dương, nhận được {val}")

        self.clip_dim = clip_dim
        self.embedding_dim = embedding_dim
        self.clip_length = clip_length
        self.projection = nn.Linear(
            in_features=clip_dim,
            out_features=clip_length * embedding_dim,
        )

    def forward(self, clip_features: Tensor) -> Tensor:
        if clip_features.ndim != 2 or clip_features.shape[1] != self.clip_dim:
            raise ValueError(
                f"clip_features phải có dạng [batch_size, {self.clip_dim}], "
                f"nhận được {tuple(clip_features.shape)}"
            )

        batch_size = clip_features.shape[0]
        projected = self.projection(clip_features)
        return projected.reshape(batch_size, self.clip_length, self.embedding_dim)

# 2. Learnable Prefix Queries & Transformer Encoder

* **Learnable Prefix Queries:** Khởi tạo tensor tham số học được $\mathbf{P} \in \mathbb{R}^{10 \times 768}$ (`nn.Parameter`), dùng chung cho mọi ảnh và mở rộng (`expand`) thành $[B, 10, 768]$.
* **Ghép chuỗi (Concatenation):** Ghép Image Tokens và Prefix Queries trên trục sequence: $[B, 10, 768] + [B, 10, 768] \rightarrow [B, 20, 768]$.
* **Full Self-Attention (Không dùng Causal Mask):** Đưa chuỗi $20$ tokens qua `nn.TransformerEncoder`. Toàn bộ token được nhìn thấy nhau hai chiều để các prefix query đọc trọn vẹn đặc trưng từ image tokens và tương tác chéo với nhau.

In [2]:
class PrefixTransformerEncoder(nn.Module):
    """Ghép Image Tokens và Learnable Prefix Queries đưa qua Transformer Encoder."""

    def __init__(
        self,
        prefix_length: int = CLIPCAP_MAPPER_PREFIX_LENGTH,
        d_model: int = 768,
        nhead: int = CLIPCAP_MAPPER_NUM_HEADS,
        num_layers: int = CLIPCAP_MAPPER_NUM_LAYERS,
        feedforward_dim: int | None = CLIPCAP_MAPPER_FEEDFORWARD_DIM,
        dropout: float = CLIPCAP_MAPPER_DROPOUT,
    ) -> None:
        super().__init__()
        if d_model % nhead != 0:
            raise ValueError(
                f"d_model ({d_model}) phải chia hết cho nhead ({nhead})"
            )

        self.prefix_length = prefix_length
        self.d_model = d_model
        dim_ff = (
            CLIPCAP_MAPPER_FEEDFORWARD_MULTIPLIER * d_model
            if feedforward_dim is None
            else feedforward_dim
        )

        # Prefix queries học được
        self.prefix_const = nn.Parameter(torch.empty(prefix_length, d_model))
        nn.init.normal_(
            self.prefix_const,
            mean=CLIPCAP_PREFIX_INIT_MEAN,
            std=CLIPCAP_PREFIX_INIT_STD,
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True,
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers,
        )

    def forward(self, image_tokens: Tensor) -> Tensor:
        batch_size = image_tokens.shape[0]

        # Expand prefix queries: [prefix_length, d_model] -> [B, prefix_length, d_model]
        prefix_queries = self.prefix_const.unsqueeze(0).expand(batch_size, -1, -1)

        # Concatenate: [B, clip_length, d_model] + [B, prefix_length, d_model] -> [B, clip_length + prefix_length, d_model]
        concat_sequence = torch.cat([image_tokens, prefix_queries], dim=1)
        return self.transformer_encoder(concat_sequence)

# 3. TransformerMapper Hoàn Chỉnh & Trích Xuất K Prefix Tokens

* **Pipeline Integration:** Nối tuần tự `ClipProjection` $\rightarrow$ `PrefixTransformerEncoder`.
* **Slicing động:** Cắt lấy chính xác $K$ prefix tokens cuối cùng bằng cú pháp `[:, self.clip_length:, :]` thay vì hard-code cố định, thu được tensor $[B, 10, 768]$ sẵn sàng ghép vào caption embeddings của GPT-2.
* **Quản lý tham số:** Bổ sung hàm `count_parameters()` để kiểm soát số lượng trọng số trainable phục vụ việc phân tích overfit trên tập dữ liệu nhỏ (Flickr8k).

In [3]:
class TransformerMapper(nn.Module):
    """
    Gộp toàn bộ pipeline:
      CLIP Features [B, clip_dim]
        -> ClipProjection -> Image Tokens [B, clip_length, embedding_dim]
        -> Transformer Encoder -> Encoded Sequence [B, clip_length + prefix_length, embedding_dim]
        -> Slicing K token cuối -> Soft Prefix [B, prefix_length, embedding_dim]
    """

    def __init__(
        self,
        clip_dim: int = 512,
        embedding_dim: int = 768,
        clip_length: int = CLIPCAP_MAPPER_CLIP_LENGTH,
        prefix_length: int = CLIPCAP_MAPPER_PREFIX_LENGTH,
        num_layers: int = CLIPCAP_MAPPER_NUM_LAYERS,
        num_heads: int = CLIPCAP_MAPPER_NUM_HEADS,
        feedforward_dim: int | None = CLIPCAP_MAPPER_FEEDFORWARD_DIM,
        dropout: float = CLIPCAP_MAPPER_DROPOUT,
    ) -> None:
        super().__init__()

        # --- 1. Validation ---
        self._validate_configuration(
            clip_dim=clip_dim,
            embedding_dim=embedding_dim,
            clip_length=clip_length,
            prefix_length=prefix_length,
            num_layers=num_layers,
            num_heads=num_heads,
            dropout=dropout,
        )

        self.clip_dim = clip_dim
        self.embedding_dim = embedding_dim
        self.clip_length = clip_length
        self.prefix_length = prefix_length
        self.feedforward_dim = (
            CLIPCAP_MAPPER_FEEDFORWARD_MULTIPLIER * embedding_dim
            if feedforward_dim is None
            else feedforward_dim
        )
        self.dropout = dropout

        # --- 2. Khởi tạo submodule ---
        self.clip_projection = ClipProjection(
            clip_dim=self.clip_dim,
            embedding_dim=self.embedding_dim,
            clip_length=self.clip_length,
        )

        self.transformer = PrefixTransformerEncoder(
            prefix_length=self.prefix_length,
            d_model=self.embedding_dim,
            nhead=num_heads,
            num_layers=num_layers,
            feedforward_dim=self.feedforward_dim,
            dropout=self.dropout,
        )

    @staticmethod
    def _validate_configuration(
        clip_dim: int,
        embedding_dim: int,
        clip_length: int,
        prefix_length: int,
        num_layers: int,
        num_heads: int,
        dropout: float,
    ) -> None:
        positive_int_params = {
            "clip_dim": clip_dim,
            "embedding_dim": embedding_dim,
            "clip_length": clip_length,
            "prefix_length": prefix_length,
            "num_layers": num_layers,
            "num_heads": num_heads,
        }
        for name, val in positive_int_params.items():
            if isinstance(val, bool) or not isinstance(val, int) or val <= 0:
                raise ValueError(f"Tham số '{name}' phải là số nguyên > 0, nhận được: {val}")

        if embedding_dim % num_heads != 0:
            raise ValueError(
                f"embedding_dim ({embedding_dim}) phải chia hết cho num_heads ({num_heads})"
            )

        if not (0.0 <= dropout < 1.0):
            raise ValueError(
                f"dropout phải nằm trong khoảng [0.0, 1.0), nhận được: {dropout}"
            )

    def forward(self, clip_features: Tensor) -> Tensor:
        # Bước 1: Project & reshape thành Image Tokens [B, clip_length, embedding_dim]
        image_tokens = self.clip_projection(clip_features)

        # Bước 2: Transformer Encoder [B, clip_length + prefix_length, embedding_dim]
        encoded_sequence = self.transformer(image_tokens)

        # Bước 3: Lấy đúng K prefix token cuối (không hard-code số 10)
        prefix_embeddings = encoded_sequence[:, self.clip_length :, :]
        return prefix_embeddings

    def count_parameters(self) -> dict[str, int]:
        """Thống kê chi tiết số lượng trainable parameters trong Mapper."""
        proj_params = sum(p.numel() for p in self.clip_projection.parameters() if p.requires_grad)
        transformer_params = sum(p.numel() for p in self.transformer.parameters() if p.requires_grad)
        total = proj_params + transformer_params
        return {
            "projection_parameters": proj_params,
            "transformer_parameters": transformer_params,
            "total_parameters": total,
        }

In [4]:
# ==============================================================================
# Sanity Check & Verification Suite
# ==============================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Khởi tạo mô hình theo baseline 4 layers
mapper = TransformerMapper(
    clip_dim=512,
    embedding_dim=768,
    clip_length=CLIPCAP_MAPPER_CLIP_LENGTH,
    prefix_length=CLIPCAP_MAPPER_PREFIX_LENGTH,
    num_layers=CLIPCAP_MAPPER_NUM_LAYERS,
    num_heads=CLIPCAP_MAPPER_NUM_HEADS,
    dropout=CLIPCAP_MAPPER_DROPOUT,
).to(device)

# 2. Thống kê tham số
param_stats = mapper.count_parameters()
print("--- Parameter Statistics ---")
for k, v in param_stats.items():
  print(f"{k:25s}: {v:,}")

# 3. Shape & Finite-Value Checks với nhiều Batch Size [1, 4, 32]
print("\n--- Multi-Batch Shape Checks ---")
mapper.eval()
with torch.no_grad():
  for bs in (1, 4, 32):
    dummy_clip = torch.randn(bs, 512, device=device)
    out_prefix = mapper(dummy_clip)

    assert out_prefix.shape == (
        bs,
        10,
        768,
    ), f"Lỗi shape tại batch size {bs}: {out_prefix.shape}"
    assert torch.isfinite(
        out_prefix
    ).all(), f"Phát hiện NaN/Inf tại batch size {bs}"
    print(f"Batch {bs:2d}: {tuple(dummy_clip.shape)} -> {tuple(out_prefix.shape)}")

--- Parameter Statistics ---
projection_parameters    : 3,939,840
transformer_parameters   : 28,359,168
total_parameters         : 32,299,008

--- Multi-Batch Shape Checks ---
Batch  1: (1, 512) -> (1, 10, 768)
Batch  4: (4, 512) -> (4, 10, 768)
Batch 32: (32, 512) -> (32, 10, 768)
